# UC-03 Tutorial: Shared-Context Automatic Match

This tutorial runs `run_use_case_03_shared_context_auto_match.py`.

Goal:
- Call backend `/shared-context/resolve`
- Observe the automatic statuses: `created` then `matched`
- Link robot observations to one canonical shared context URI

Prerequisites:
- Open this notebook from inside the SEGB repository.
- Start backend API at `http://localhost:5000`.
- Python env:
  - `./.venv/bin/python -m pip install -e packages/semantic_log_generator`
  - `./.venv/bin/python -m pip install pydantic`
- Run the backend preflight cell below before executing the main use-case cell.
- UI routes can be opened on:
  - Production: `http://localhost:8080`
  - Development: `http://localhost:5173`


In [ ]:
# Ensure repository root is available in sys.path
import sys
from pathlib import Path

repo_root = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / 'examples').exists() and (candidate / 'packages' / 'semantic_log_generator').exists():
        repo_root = candidate
        break

if repo_root is None:
    raise RuntimeError('Repository root not found. Open this notebook inside the SEGB repository.')

extra_paths = [
    repo_root,
    repo_root / 'packages' / 'semantic_log_generator' / 'src',
    repo_root / 'apps' / 'backend' / 'src',
]
for path in extra_paths:
    path_str = str(path)
    if path.exists() and path_str not in sys.path:
        sys.path.insert(0, path_str)

print(f'Using repo root: {repo_root}')


In [ ]:
# Backend preflight (health checks)
import json
import urllib.error
import urllib.request

API_URL = 'http://localhost:5000'
REQUIRE_BACKEND = True


def _get_json(url: str) -> dict:
    with urllib.request.urlopen(url, timeout=5) as response:
        payload = response.read().decode('utf-8')
    return json.loads(payload)


def check_backend_health(api_url: str, *, require: bool) -> None:
    base = api_url.rstrip('/')
    live_url = f"{base}/healthz/live"
    ready_url = f"{base}/healthz/ready"

    try:
        live = _get_json(live_url)
        ready = _get_json(ready_url)
    except (urllib.error.URLError, TimeoutError, json.JSONDecodeError) as exc:
        message = f"Backend not reachable at {api_url}: {exc}"
        if require:
            raise RuntimeError(message) from exc
        print('WARNING:', message)
        return

    live_ok = bool(live.get('live'))
    ready_ok = bool(ready.get('ready')) and bool(ready.get('virtuoso'))

    print('healthz/live:', live)
    print('healthz/ready:', ready)

    if not (live_ok and ready_ok):
        message = f"Backend unhealthy at {api_url} (live_ok={live_ok}, ready_ok={ready_ok})"
        if require:
            raise RuntimeError(message)
        print('WARNING:', message)
        return

    print(f"Backend preflight OK -> {api_url}")


check_backend_health(API_URL, require=REQUIRE_BACKEND)


In [ ]:
from examples.simulations.http_helpers import ApiConfig
from examples.simulations.run_use_case_03_shared_context_auto_match import run_shared_context_auto_match_use_case

API_URL = 'http://localhost:5000'
TOKEN = None  # set JWT string if auth is enabled

api_config = ApiConfig(base_url=API_URL, token=TOKEN, timeout_seconds=20.0, verify_tls=True)
result = run_shared_context_auto_match_use_case(api_config=api_config)

print('Use case: UC-03 shared-context auto-match')
print('Shared context URI:', result.shared_context_uri)
print('First status:', result.first_resolution.status)
print('Second status:', result.second_resolution.status)
print('Triples generated:', len(result.graph))


## Optional: Publish to Backend

```bash
./.venv/bin/python -m examples.simulations.run_use_case_03_shared_context_auto_match \
  --publish-url http://localhost:5000 \
  --no-print-ttl
```

## What to Verify in the Web UI

Use the same route on prod (`http://localhost:8080`) or dev (`http://localhost:5173`).

1. Open `/shared-context`.
2. Check resolver statistics and confirm contexts were created/matched.
3. Open `/kg-graph` and find both observations linked to the same shared context node.
